# Análisis Topológico de Servicios Médicos en Iztapalapa
## Persistencia Homológica sobre datos DENUE 2025

**Alcaldía:** Iztapalapa, CDMX  
**Sector:** Salud general (códigos SCIAN 621111, 621112, 621115, 621116, 622111, 622112)  
**Herramientas:** Ripser, GUDHI, Folium, GeoPandas

---
### Estructura del análisis
1. Carga y preparación de datos
2. Mapa interactivo (Folium)
3. Complejos Simpliciales: Vietoris-Rips y Alpha (≈ Čech)
4. Diagramas de barras (Barcodes) — H0 y H1
5. Diagramas de persistencia
6. Análisis por sector: Público vs Privado
7. Zonas de cobertura y vacíos persistentes
8. Conclusiones económicas

In [ ]:
# ============================================================
# SECCIÓN 0: IMPORTS Y CONFIGURACIÓN
# ============================================================
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from sklearn.neighbors import BallTree
from scipy.spatial.distance import pdist, squareform
from itertools import combinations
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# TDA
from ripser import ripser
from persim import plot_diagrams
import gudhi

# Mapas interactivos
import folium
from folium.plugins import MarkerCluster, HeatMap

# Carpeta de salida
figures_dir = Path('outputs/figures')
figures_dir.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print('✓ Librerías cargadas')

In [ ]:
# ============================================================
# SECCIÓN 1: CARGA Y PREPARACIÓN
# ============================================================

df = pd.read_csv('outputs/salud_general_iztapalapa_enriquecida.csv', encoding='latin1')
df['latitud']  = pd.to_numeric(df['latitud'],  errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
df = df.dropna(subset=['latitud', 'longitud']).copy()

# GeoDataFrame en WGS84
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['longitud'], df['latitud']),
    crs='EPSG:4326'
)

# Proyectar a metros (UTM zona 14N para CDMX)
gdf_m      = gdf.to_crs('EPSG:32614')
coords_m   = np.array([(g.x, g.y) for g in gdf_m.geometry])
coords_c   = coords_m - coords_m.mean(axis=0)  # centradas para TDA

# Límites de alcaldías
alcaldias   = gpd.read_file('data/limite-de-las-alcaldas.json').to_crs('EPSG:4326')
iztapalapa  = alcaldias[alcaldias['NOMGEO'] == 'Iztapalapa']
iztapalapa_m = iztapalapa.to_crs('EPSG:32614')

# Paleta de colores por tipo
COLOR_TIPO = {
    'Consultorio general privado' : '#E74C3C',
    'Consultorio general público'  : '#3498DB',
    'Clínica privada'              : '#F39C12',
    'Clínica pública'              : '#27AE60',
    'Hospital general privado'     : '#8E44AD',
    'Hospital general público'     : '#1ABC9C',
}

COLOR_SECTOR = {'Privado': '#E74C3C', 'Público': '#27AE60'}

print(f'✓ Datos cargados: {len(df)} establecimientos')
print(f'  Sector privado: {(df["sector"]=="Privado").sum()}  |  Sector público: {(df["sector"]=="Público").sum()}')
print(f'  Coordenadas: lat [{df["latitud"].min():.4f}, {df["latitud"].max():.4f}]')
print(f'               lon [{df["longitud"].min():.4f}, {df["longitud"].max():.4f}]')

In [ ]:
# ============================================================
# SECCIÓN 2A: MAPA INTERACTIVO — DISTRIBUCIÓN POR TIPO
# ============================================================

centro = [df['latitud'].mean(), df['longitud'].mean()]
m = folium.Map(location=centro, zoom_start=13, tiles='CartoDB positron')

# Límite de Iztapalapa
folium.GeoJson(
    iztapalapa.__geo_interface__,
    style_function=lambda x: {'fillColor': 'none', 'color': '#2C3E50', 'weight': 2.5}
).add_to(m)

# Clusters por tipo
clusters = {tipo: MarkerCluster(name=tipo) for tipo in COLOR_TIPO}
for _, row in df.iterrows():
    tipo  = row['tipo_establecimiento']
    color = COLOR_TIPO.get(tipo, '#999')
    popup_text = (
        f"<b>{row['nom_estab']}</b><br>"
        f"Tipo: {tipo}<br>"
        f"Personal: {row['per_ocu']}<br>"
        f"CP: {int(row['cod_postal']) if pd.notna(row['cod_postal']) else 'N/D'}"
    )
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.75,
        popup=folium.Popup(popup_text, max_width=250)
    ).add_to(clusters[tipo])

for cluster in clusters.values():
    cluster.add_to(m)

folium.LayerControl().add_to(m)
m.save('outputs/mapa_interactivo_tipos.html')
print('✓ Mapa guardado: outputs/mapa_interactivo_tipos.html')
m

In [ ]:
# ============================================================
# SECCIÓN 2B: MAPA INTERACTIVO — HEATMAP DE DENSIDAD
# ============================================================

m2 = folium.Map(location=centro, zoom_start=13, tiles='CartoDB dark_matter')

# Límite
folium.GeoJson(
    iztapalapa.__geo_interface__,
    style_function=lambda x: {'fillColor': 'none', 'color': 'white', 'weight': 2}
).add_to(m2)

# Heatmap general
heat_data = df[['latitud', 'longitud']].values.tolist()
HeatMap(heat_data, radius=15, blur=12, min_opacity=0.4).add_to(
    folium.FeatureGroup(name='Densidad general').add_to(m2)
)

# Heatmap sólo hospitales (mayor peso)
hospitales = df[df['tipo_establecimiento'].str.contains('Hospital')]
heat_hosp  = hospitales[['latitud', 'longitud']].values.tolist()
HeatMap(heat_hosp, radius=20, blur=15, min_opacity=0.5, gradient={'0.4': 'blue', '0.6': 'cyan', '1': 'red'}).add_to(
    folium.FeatureGroup(name='Densidad hospitales').add_to(m2)
)

folium.LayerControl().add_to(m2)
m2.save('outputs/mapa_heatmap_densidad.html')
print('✓ Mapa guardado: outputs/mapa_heatmap_densidad.html')
m2

In [ ]:
# ============================================================
# SECCIÓN 3A: COMPLEJO VIETORIS-RIPS — VISUALIZACIÓN MEJORADA
# ============================================================
# Usamos submuestra para que los triángulos sean legibles visualmente.
# La persistencia se calculará sobre todos los puntos.

D = squareform(pdist(coords_m))

def plot_vr_complex(coords, boundary_gdf, radio, ax, title, show_triangles=True, alpha_tri=0.08):
    """Dibuja complejo VR sobre un eje dado."""
    boundary_gdf.plot(ax=ax, facecolor='none', edgecolor='#2C3E50', linewidth=1.5)
    D_local = squareform(pdist(coords))
    n = len(coords)

    if show_triangles:
        for i, j, k in combinations(range(n), 3):
            if D_local[i,j] <= radio and D_local[i,k] <= radio and D_local[j,k] <= radio:
                tri = np.array([coords[i], coords[j], coords[k]])
                ax.fill(tri[:,0], tri[:,1], color='steelblue', alpha=alpha_tri, zorder=1)

    for i, j in combinations(range(n), 2):
        if D_local[i,j] <= radio:
            ax.plot([coords[i,0], coords[j,0]], [coords[i,1], coords[j,1]],
                    color='steelblue', alpha=0.3, linewidth=0.7, zorder=2)

    ax.scatter(coords[:,0], coords[:,1], s=20, color='#E74C3C', zorder=4, alpha=0.9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

# --- Submuestra de 80 puntos para visualización clara ---
np.random.seed(42)
idx_sub  = np.random.choice(len(coords_m), size=min(80, len(coords_m)), replace=False)
coords_sub = coords_m[idx_sub]

radios = [300, 700, 1200, 2000]
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

for ax, r in zip(axes, radios):
    plot_vr_complex(coords_sub, iztapalapa_m, r, ax,
                    title=f'VR  r = {r} m',
                    show_triangles=(r <= 1200),
                    alpha_tri=0.07)

fig.suptitle('Evolución del complejo Vietoris–Rips — Servicios médicos Iztapalapa\n(submuestra n=80)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(figures_dir / '11_vr_evolucion_4radios.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 11_vr_evolucion_4radios.png')

In [ ]:
# ============================================================
# SECCIÓN 3B: COMPLEJO ALPHA (EQUIVALENTE A ČECH EN HOMOLOGÍA)
# ============================================================
# El complejo Alpha es un subconjunto del complejo de Čech con
# la misma homología. Es más eficiente computacionalmente.
# GUDHI lo construye sobre la triangulación de Delaunay.

# Construir Alpha Complex con GUDHI sobre todos los puntos
alpha_complex  = gudhi.AlphaComplex(points=coords_c.tolist())
simplex_tree_a = alpha_complex.create_simplex_tree()

print(f'Complejo Alpha construido:')
print(f'  Vértices (0-simplices): {sum(1 for s,_ in simplex_tree_a.get_filtration() if len(s)==1)}')
print(f'  Aristas  (1-simplices): {sum(1 for s,_ in simplex_tree_a.get_filtration() if len(s)==2)}')
print(f'  Triángulos (2-simplices): {sum(1 for s,_ in simplex_tree_a.get_filtration() if len(s)==3)}')

# Persistencia con Alpha Complex
simplex_tree_a.compute_persistence()
dgm_alpha_h0 = simplex_tree_a.persistence_intervals_in_dimension(0)
dgm_alpha_h1 = simplex_tree_a.persistence_intervals_in_dimension(1)

# --- Visualizar complejo Alpha (submuestra) ---
alpha_sub  = gudhi.AlphaComplex(points=coords_sub.tolist())
st_sub     = alpha_sub.create_simplex_tree()

# Radio umbral equivalente a 700m^2
ALPHA_MAX = 700**2

fig, ax = plt.subplots(figsize=(10, 10))
iztapalapa_m.plot(ax=ax, facecolor='#F8F9FA', edgecolor='#2C3E50', linewidth=1.5)

for simplex, filt in st_sub.get_filtration():
    if filt > ALPHA_MAX:
        continue
    if len(simplex) == 3:
        tri = coords_sub[simplex]
        ax.fill(tri[:,0], tri[:,1], color='#3498DB', alpha=0.08, zorder=1)
    elif len(simplex) == 2:
        p1, p2 = coords_sub[simplex[0]], coords_sub[simplex[1]]
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='#2980B9', alpha=0.4, lw=0.8, zorder=2)

ax.scatter(coords_sub[:,0], coords_sub[:,1], s=25, color='#C0392B', zorder=4)
ax.set_title(f'Complejo Alpha — Iztapalapa (α² ≤ {ALPHA_MAX:,} m²)\n(submuestra n=80)',
             fontsize=13, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')
plt.tight_layout()
plt.savefig(figures_dir / '12_complejo_alpha_iztapalapa.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 12_complejo_alpha_iztapalapa.png')

In [ ]:
# ============================================================
# SECCIÓN 4: PERSISTENCIA HOMOLÓGICA COMPLETA (RIPSER)
# ============================================================
# Usamos todos los 563 puntos con umbral de 5 km

result   = ripser(coords_c, maxdim=1, thresh=5000)
dgms     = result['dgms']
dgm_h0   = dgms[0]  # H0: componentes conectadas
dgm_h1   = dgms[1]  # H1: ciclos / agujeros

print(f'H0 — Componentes conectadas: {len(dgm_h0)} características')
print(f'H1 — Ciclos (agujeros):      {len(dgm_h1)} características')

# Filtrar finitos (excluir la componente que vive hasta infinito en H0)
h0_finito = dgm_h0[dgm_h0[:,1] < np.inf]
h1_finito = dgm_h1[dgm_h1[:,1] < np.inf]

print(f'\nH0 finitos: {len(h0_finito)}')
print(f'H0 — Nacimiento máximo  : {h0_finito[:,0].max():.1f} m')
print(f'H0 — Muerte máxima      : {h0_finito[:,1].max():.1f} m')
print(f'H0 — Persistencia máxima: {(h0_finito[:,1]-h0_finito[:,0]).max():.1f} m')

if len(h1_finito) > 0:
    print(f'\nH1 finitos: {len(h1_finito)}')
    print(f'H1 — Nacimiento mín/máx : {h1_finito[:,0].min():.1f} / {h1_finito[:,0].max():.1f} m')
    print(f'H1 — Muerte mín/máx     : {h1_finito[:,1].min():.1f} / {h1_finito[:,1].max():.1f} m')
    pers_h1 = h1_finito[:,1] - h1_finito[:,0]
    print(f'H1 — Persistencia máxima: {pers_h1.max():.1f} m (ciclo más persistente)')

In [ ]:
# ============================================================
# SECCIÓN 4A: DIAGRAMA DE BARRAS (BARCODE) — H0 y H1
# ============================================================

def plot_barcode(dgm, dim, ax, max_disp=40, color='steelblue', title=''):
    """Barcode diagram para una dimensión homológica."""
    # Separar finitos e infinitos
    finitos  = dgm[dgm[:,1] < np.inf]
    inf_mask = dgm[:,1] == np.inf

    # Ordenar por persistencia descendente
    pers = finitos[:,1] - finitos[:,0]
    idx  = np.argsort(-pers)
    finitos_ord = finitos[idx]

    # Limitar a max_disp barras
    finitos_show = finitos_ord[:max_disp]
    n_fin = len(finitos_show)
    n_inf = inf_mask.sum()

    # Valor máximo para truncar barras infinitas
    x_max = finitos[:,1].max() * 1.05 if len(finitos) > 0 else 5000

    y = 0
    for birth, death in finitos_show:
        p = death - birth
        ax.barh(y, p, left=birth, height=0.7,
                color=color, alpha=0.6 + 0.4*(p/pers.max()), edgecolor='white', lw=0.3)
        y += 1

    # Barras infinitas
    if dim == 0 and n_inf > 0:
        birth_inf = dgm[inf_mask, 0]
        for b in birth_inf:
            ax.barh(y, x_max - b, left=b, height=0.7,
                    color='#E74C3C', alpha=0.8, edgecolor='white', lw=0.3)
            y += 1

    ax.set_xlabel('Radio de filtración (metros)', fontsize=11)
    ax.set_ylabel(f'Características H{dim}', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='black', lw=0.8)
    if dim == 0:
        ax.legend(
            handles=[
                mpatches.Patch(color=color, alpha=0.7, label=f'H0 finitas ({n_fin})'),
                mpatches.Patch(color='#E74C3C', alpha=0.8, label=f'H0 persistente ({n_inf})')
            ], fontsize=9
        )
    else:
        ax.legend(
            handles=[mpatches.Patch(color=color, alpha=0.7, label=f'H1 ciclos ({n_fin})')],
            fontsize=9
        )

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(16, 7))

plot_barcode(dgm_h0, dim=0, ax=ax0, max_disp=30,
             color='#2980B9',
             title='Barcode H₀ — Componentes conectadas\n(servicios médicos Iztapalapa)')

plot_barcode(dgm_h1, dim=1, ax=ax1, max_disp=30,
             color='#E67E22',
             title='Barcode H₁ — Ciclos / Vacíos de cobertura\n(servicios médicos Iztapalapa)')

plt.tight_layout()
plt.savefig(figures_dir / '13_barcodes_h0_h1.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 13_barcodes_h0_h1.png')

In [ ]:
# ============================================================
# SECCIÓN 4B: DIAGRAMA DE PERSISTENCIA MEJORADO
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Diagrama de dispersión estándar ---
ax = axes[0]
h0_f = dgm_h0[dgm_h0[:,1] < np.inf]
h1_f = dgm_h1[dgm_h1[:,1] < np.inf]
h0_i = dgm_h0[dgm_h0[:,1] == np.inf]

lim = max(h0_f[:,1].max() if len(h0_f) else 0,
          h1_f[:,1].max() if len(h1_f) else 0) * 1.05

ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='Diagonal')
ax.scatter(h0_f[:,0], h0_f[:,1], s=40, color='#2980B9', alpha=0.7, label=f'H₀ ({len(h0_f)} finitas)', zorder=3)
ax.scatter(h0_i[:,0], [lim*0.98]*len(h0_i), s=80, color='#C0392B',
           marker='^', label=f'H₀ (∞)', zorder=4)
ax.scatter(h1_f[:,0], h1_f[:,1], s=60, color='#E67E22', alpha=0.85,
           marker='D', label=f'H₁ ({len(h1_f)} ciclos)', zorder=3)

# Anotar los 3 ciclos H1 más persistentes
if len(h1_f) > 0:
    pers1 = h1_f[:,1] - h1_f[:,0]
    top3  = np.argsort(-pers1)[:3]
    for rank, idx in enumerate(top3):
        b, d = h1_f[idx]
        ax.annotate(f'C{rank+1} ({(d-b):.0f}m)', (b, d),
                    xytext=(b+50, d+50), fontsize=8, color='#D35400',
                    arrowprops=dict(arrowstyle='->', color='#D35400', lw=0.8))

ax.set_xlabel('Nacimiento (radio, m)', fontsize=11)
ax.set_ylabel('Muerte (radio, m)', fontsize=11)
ax.set_title('Diagrama de persistencia\nH₀ y H₁', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(-50, lim)
ax.set_ylim(-50, lim)
ax.grid(True, alpha=0.3)

# --- Persistencia vs índice (ranking) ---
ax2 = axes[1]
if len(h1_f) > 0:
    pers1_sorted = np.sort(h1_f[:,1] - h1_f[:,0])[::-1]
    ax2.bar(range(len(pers1_sorted)), pers1_sorted,
            color='#E67E22', alpha=0.75, edgecolor='white', lw=0.4)
    ax2.axhline(y=np.mean(pers1_sorted), color='red', ls='--', lw=1.5,
                label=f'Media: {np.mean(pers1_sorted):.0f} m')
    ax2.set_xlabel('Ciclo H₁ (ordenado por persistencia)', fontsize=11)
    ax2.set_ylabel('Persistencia (muerte − nacimiento, m)', fontsize=11)
    ax2.set_title('Persistencia de ciclos H₁\n(vacíos de cobertura)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / '14_diagrama_persistencia_mejorado.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 14_diagrama_persistencia_mejorado.png')

In [ ]:
# ============================================================
# SECCIÓN 5: ANÁLISIS POR SECTOR — PÚBLICO VS PRIVADO
# ============================================================

def get_coords_sector(df_full, gdf_m_full, sector):
    idx = df_full[df_full['sector'] == sector].index
    sub = gdf_m_full.loc[idx]
    c = np.array([(g.x, g.y) for g in sub.geometry])
    return c - c.mean(axis=0)

# Mantener índices sincronizados
df_idx    = df.reset_index(drop=True)
gdf_m_idx = gdf_m.reset_index(drop=True)

coords_priv = get_coords_sector(df_idx, gdf_m_idx, 'Privado')
coords_pub  = get_coords_sector(df_idx, gdf_m_idx, 'Público')

print(f'Sector privado: {len(coords_priv)} puntos')
print(f'Sector público: {len(coords_pub)} puntos')

# Persistencia por sector
res_priv = ripser(coords_priv, maxdim=1, thresh=5000)
res_pub  = ripser(coords_pub,  maxdim=1, thresh=5000)

dgm_priv = res_priv['dgms']
dgm_pub  = res_pub['dgms']

# --- Visualización comparativa ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, (dgms, label, color) in enumerate([
    (dgm_priv, 'Sector Privado (n=492)', '#E74C3C'),
    (dgm_pub,  'Sector Público (n=71)',  '#27AE60')
]):
    # Barcode H1
    ax_bc = axes[0, col]
    h1 = dgms[1]
    h1_f = h1[h1[:,1] < np.inf]
    if len(h1_f) > 0:
        pers = h1_f[:,1] - h1_f[:,0]
        idx_s = np.argsort(-pers)
        for y_pos, idx_i in enumerate(idx_s[:25]):
            b, d = h1_f[idx_i]
            ax_bc.barh(y_pos, d-b, left=b, height=0.7,
                       color=color, alpha=0.5+0.5*(pers[idx_i]/pers.max()),
                       edgecolor='white', lw=0.3)
    ax_bc.set_title(f'Barcode H₁\n{label}', fontsize=11, fontweight='bold')
    ax_bc.set_xlabel('Radio (m)')
    ax_bc.set_ylabel('Ciclos H₁')
    ax_bc.grid(axis='x', alpha=0.3)

    # Diagrama de persistencia
    ax_pd = axes[1, col]
    h0 = dgms[0]; h1 = dgms[1]
    h0_f = h0[h0[:,1] < np.inf]; h1_f = h1[h1[:,1] < np.inf]
    lim_l = max(h0_f[:,1].max() if len(h0_f) else 0,
                h1_f[:,1].max() if len(h1_f) else 0) * 1.05
    ax_pd.plot([0,lim_l],[0,lim_l],'k--',lw=1,alpha=0.4)
    ax_pd.scatter(h0_f[:,0], h0_f[:,1], s=30, color='#2980B9', alpha=0.6, label='H₀')
    ax_pd.scatter(h1_f[:,0], h1_f[:,1], s=50, color=color, alpha=0.8, marker='D', label='H₁')
    ax_pd.set_title(f'Diagrama de persistencia\n{label}', fontsize=11, fontweight='bold')
    ax_pd.set_xlabel('Nacimiento (m)')
    ax_pd.set_ylabel('Muerte (m)')
    ax_pd.legend(fontsize=9)
    ax_pd.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / '15_comparacion_sector_publico_privado.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 15_comparacion_sector_publico_privado.png')

In [ ]:
# ============================================================
# SECCIÓN 6: EVOLUCIÓN DE COMPONENTES CONECTADAS (H0)
# ============================================================
# Mostramos cómo el número de componentes conectadas disminuye
# conforme aumenta el radio — esto refleja la conectividad de la red.

# Usar los valores de muerte de H0 (radio en que dos componentes se unen)
h0_f_sorted = np.sort(dgm_h0[dgm_h0[:,1] < np.inf, 1])

# Número de componentes en función del radio
n_puntos = len(coords_c)
radios_eval = np.linspace(0, h0_f_sorted[-1] * 1.1, 500)
n_componentes = np.array([n_puntos - np.sum(h0_f_sorted <= r) for r in radios_eval])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Curva de componentes
ax1.plot(radios_eval, n_componentes, color='#2980B9', lw=2.5)
ax1.fill_between(radios_eval, n_componentes, alpha=0.15, color='#2980B9')

# Marcar radios relevantes
for r_mark, label in [(300,'300 m'), (500,'500 m'), (1000,'1 km'), (2000,'2 km')]:
    if r_mark <= radios_eval[-1]:
        n_c = n_puntos - np.sum(h0_f_sorted <= r_mark)
        ax1.axvline(x=r_mark, color='gray', ls='--', lw=1, alpha=0.7)
        ax1.annotate(f'{label}\n({n_c} comp.)', (r_mark, n_c),
                     xytext=(r_mark+50, n_c+5), fontsize=8, color='#2C3E50')

ax1.set_xlabel('Radio de filtración (metros)', fontsize=11)
ax1.set_ylabel('Número de componentes conectadas', fontsize=11)
ax1.set_title('Evolución de componentes conectadas H₀\n(todos los servicios médicos)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# Distribución de radios de conexión
ax2.hist(h0_f_sorted, bins=40, color='#2980B9', alpha=0.7, edgecolor='white')
ax2.axvline(np.median(h0_f_sorted), color='red', ls='--', lw=2,
            label=f'Mediana: {np.median(h0_f_sorted):.0f} m')
ax2.axvline(np.mean(h0_f_sorted), color='orange', ls='--', lw=2,
            label=f'Media: {np.mean(h0_f_sorted):.0f} m')
ax2.set_xlabel('Radio de unión de componentes (metros)', fontsize=11)
ax2.set_ylabel('Frecuencia', fontsize=11)
ax2.set_title('Distribución de radios de conexión H₀\n(distancias críticas de conectividad)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / '16_evolucion_componentes_h0.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 16_evolucion_componentes_h0.png')

In [ ]:
# ============================================================
# SECCIÓN 7A: MAPA DE ZONAS DE COBERTURA POR RADIO
# ============================================================
# Visualizamos la cobertura potencial de cada establecimiento
# como un círculo de radio fijo — equivalente a la bola en el
# complejo de Čech. Radio = mitad del umbral de conectividad.

RADIO_COBERTURA = 500  # metros — radio de atención estimado

m3 = folium.Map(location=centro, zoom_start=13, tiles='CartoDB positron')

# Límite de alcaldía
folium.GeoJson(
    iztapalapa.__geo_interface__,
    style_function=lambda x: {'fillColor': 'none', 'color': '#2C3E50', 'weight': 2.5}
).add_to(m3)

# Círculos de cobertura (transparentes) por sector
fg_priv = folium.FeatureGroup(name='Cobertura Privada').add_to(m3)
fg_pub  = folium.FeatureGroup(name='Cobertura Pública').add_to(m3)

for _, row in df.iterrows():
    color = '#E74C3C' if row['sector'] == 'Privado' else '#27AE60'
    fg = fg_priv if row['sector'] == 'Privado' else fg_pub
    folium.Circle(
        location=[row['latitud'], row['longitud']],
        radius=RADIO_COBERTURA,
        color=color,
        fill=True,
        fill_opacity=0.04,
        weight=0.3
    ).add_to(fg)
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.9,
        popup=f"{row['nom_estab']} — {row['tipo_establecimiento']}"
    ).add_to(fg)

folium.LayerControl().add_to(m3)
m3.save('outputs/mapa_cobertura_500m.html')
print(f'✓ Mapa de cobertura guardado: outputs/mapa_cobertura_500m.html')
m3

In [ ]:
# ============================================================
# SECCIÓN 7B: IDENTIFICACIÓN DE VACÍOS DE COBERTURA PERSISTENTES
# ============================================================
# Los ciclos H1 más persistentes indican regiones donde existe
# un anillo de servicios con un vacío al interior — posibles
# zonas sin acceso a salud general.

h1_finito = dgm_h1[dgm_h1[:,1] < np.inf].copy()
if len(h1_finito) > 0:
    pers_h1 = h1_finito[:,1] - h1_finito[:,0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Scatter: radio de nacimiento vs persistencia ---
    ax = axes[0]
    sc = ax.scatter(h1_finito[:,0], pers_h1,
                    c=pers_h1, cmap='YlOrRd', s=80, alpha=0.85,
                    edgecolors='gray', lw=0.5, zorder=3)
    plt.colorbar(sc, ax=ax, label='Persistencia (m)')

    # Línea umbral: ciclos con persistencia > media+std son "significativos"
    umbral_sig = pers_h1.mean() + pers_h1.std()
    ax.axhline(umbral_sig, color='red', ls='--', lw=1.5,
               label=f'Umbral (μ+σ = {umbral_sig:.0f} m)')

    n_sig = (pers_h1 >= umbral_sig).sum()
    ax.set_xlabel('Radio de nacimiento del ciclo (m)', fontsize=11)
    ax.set_ylabel('Persistencia del ciclo (m)', fontsize=11)
    ax.set_title(f'Ciclos H₁ y su persistencia\n({n_sig} ciclos significativos, umbral={umbral_sig:.0f} m)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # --- Histograma de persistencias ---
    ax2 = axes[1]
    ax2.hist(pers_h1, bins=25, color='#E67E22', alpha=0.75, edgecolor='white')
    ax2.axvline(pers_h1.mean(), color='red', ls='--', lw=2,
                label=f'Media: {pers_h1.mean():.0f} m')
    ax2.axvline(umbral_sig, color='darkred', ls=':', lw=2,
                label=f'Umbral (μ+σ): {umbral_sig:.0f} m')
    ax2.set_xlabel('Persistencia del ciclo H₁ (metros)', fontsize=11)
    ax2.set_ylabel('Número de ciclos', fontsize=11)
    ax2.set_title('Distribución de persistencias H₁\n(tamaño de vacíos de cobertura)',
                  fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(figures_dir / '17_vacios_cobertura_h1.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('✓ Guardada: 17_vacios_cobertura_h1.png')
    print(f'\n  Ciclos H1 significativos (persistencia > {umbral_sig:.0f} m): {n_sig}')
    print(f'  Radio de nacimiento promedio de ciclos sig.: {h1_finito[pers_h1>=umbral_sig, 0].mean():.0f} m')
else:
    print('No hay ciclos H1 finitos detectados — la nube es muy densa.')

In [ ]:
# ============================================================
# SECCIÓN 7C: MAPA ESTÁTICO — DENSIDAD + AISLAMIENTO + HOSPITALES
# ============================================================

from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(12, 12))

# KDE de densidad de puntos
lon_arr = df['longitud'].values
lat_arr = df['latitud'].values
kde = gaussian_kde(np.vstack([lon_arr, lat_arr]), bw_method=0.03)

xmin, xmax = lon_arr.min()-0.005, lon_arr.max()+0.005
ymin, ymax = lat_arr.min()-0.005, lat_arr.max()+0.005
xx, yy = np.mgrid[xmin:xmax:200j, ymin:ymax:200j]
zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)

# Fondo de densidad
im = ax.contourf(xx, yy, zz, levels=15, cmap='YlOrRd', alpha=0.55, zorder=1)
plt.colorbar(im, ax=ax, label='Densidad de establecimientos (KDE)', shrink=0.7)

# Límite de alcaldía
iztapalapa.plot(ax=ax, facecolor='none', edgecolor='#2C3E50', linewidth=2, zorder=2)

# Establecimientos aislados
aislados = df[df['es_aislado'] == True]
no_aisl  = df[df['es_aislado'] == False]
ax.scatter(no_aisl['longitud'], no_aisl['latitud'],
           s=25, color='#2C3E50', alpha=0.3, zorder=3, label='Establecimiento')
ax.scatter(aislados['longitud'], aislados['latitud'],
           s=80, color='#8E44AD', alpha=0.85, zorder=4,
           edgecolors='white', lw=0.8, marker='o', label='Aislado (p90)')

# Hospitales
hosp = df[df['tipo_establecimiento'].str.contains('Hospital')]
for tipo, sym in [('Hospital general privado','s'),('Hospital general público','*')]:
    sub = hosp[hosp['tipo_establecimiento']==tipo]
    ax.scatter(sub['longitud'], sub['latitud'],
               s=180 if sym=='*' else 120,
               color=COLOR_TIPO[tipo], alpha=0.95,
               marker=sym, zorder=5, edgecolors='black', lw=0.8, label=tipo)

ax.set_title('Densidad, aislamiento y hospitales — Servicios médicos Iztapalapa',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Longitud', fontsize=11)
ax.set_ylabel('Latitud', fontsize=11)
ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(figures_dir / '18_densidad_aislamiento_hospitales.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Guardada: 18_densidad_aislamiento_hospitales.png')

In [ ]:
# ============================================================
# SECCIÓN 8: TABLA RESUMEN DE PERSISTENCIA HOMOLÓGICA
# ============================================================

h0_f = dgm_h0[dgm_h0[:,1] < np.inf]
h1_f = dgm_h1[dgm_h1[:,1] < np.inf]

print('=' * 65)
print(' RESUMEN DE PERSISTENCIA HOMOLÓGICA — IZTAPALAPA')
print('=' * 65)
print(f'\n Puntos analizados: {len(coords_c)}')

print(f'\n H₀ — COMPONENTES CONECTADAS')
print(f'   Total características:    {len(dgm_h0)}')
print(f'   Componente persistente:   1 (la alcaldía completa)')
print(f'   Características finitas:  {len(h0_f)}')
if len(h0_f) > 0:
    pers0 = h0_f[:,1] - h0_f[:,0]
    print(f'   Persistencia media:       {pers0.mean():.1f} m')
    print(f'   Persistencia máxima:      {pers0.max():.1f} m')
    print(f'   Radio de conectividad p50:{np.median(h0_f[:,1]):.1f} m')
    print(f'   → A r={np.median(h0_f[:,1]):.0f} m el 50% de puntos están conectados.')

print(f'\n H₁ — CICLOS / VACÍOS DE COBERTURA')
print(f'   Total ciclos:             {len(dgm_h1)}')
print(f'   Ciclos finitos:           {len(h1_f)}')
if len(h1_f) > 0:
    pers1 = h1_f[:,1] - h1_f[:,0]
    umbral_s = pers1.mean() + pers1.std()
    n_sig_s  = (pers1 >= umbral_s).sum()
    print(f'   Persistencia media:       {pers1.mean():.1f} m')
    print(f'   Persistencia máxima:      {pers1.max():.1f} m (vacío más grande)')
    print(f'   Ciclos significativos:    {n_sig_s} (persistencia > {umbral_s:.0f} m)')
    print(f'\n   Top 5 ciclos más persistentes:')
    top5 = np.argsort(-pers1)[:5]
    for i, idx in enumerate(top5):
        b, d = h1_f[idx]
        print(f'   {i+1}. Nace r={b:.0f} m → muere r={d:.0f} m | persistencia={d-b:.0f} m')

print(f'\n INTERPRETACIÓN ECONÓMICA')
print('-' * 65)
print(f'   El 87.4% de establecimientos son privados — fuerte predominio')
print(f'   del mercado privado en salud general de Iztapalapa.')
if len(h1_f) > 0:
    print(f'   Los {n_sig_s} ciclos H₁ significativos identifican zonas donde')
    print(f'   existe un anillo de servicios pero el interior carece de')
    print(f'   cobertura directa — candidatos a despliegue de servicios públicos.')
print('=' * 65)

In [ ]:
# ============================================================
# SECCIÓN 9: MAPA INTERACTIVO — AISLADOS Y HOSPITALES
# ============================================================

m4 = folium.Map(location=centro, zoom_start=13, tiles='CartoDB positron')

folium.GeoJson(
    iztapalapa.__geo_interface__,
    style_function=lambda x: {'fillColor': 'none', 'color': '#2C3E50', 'weight': 2.5}
).add_to(m4)

fg_normal   = folium.FeatureGroup(name='Establecimientos normales').add_to(m4)
fg_aislados = folium.FeatureGroup(name='Aislados (p90)').add_to(m4)
fg_hosp     = folium.FeatureGroup(name='Hospitales').add_to(m4)

for _, row in df.iterrows():
    es_hosp = 'Hospital' in row['tipo_establecimiento']
    es_aisl = row['es_aislado']
    color   = COLOR_TIPO.get(row['tipo_establecimiento'], '#999')
    popup   = (f"<b>{row['nom_estab']}</b><br>"
               f"{row['tipo_establecimiento']}<br>"
               f"Personal: {row['per_ocu']}<br>"
               f"Dist. vecino más cercano: {row['distancia_vecino_mas_cercano_m']:.0f} m")

    if es_hosp:
        folium.Marker(
            location=[row['latitud'], row['longitud']],
            popup=folium.Popup(popup, max_width=250),
            icon=folium.Icon(color='red' if row['sector']=='Privado' else 'green',
                             icon='plus-sign', prefix='glyphicon')
        ).add_to(fg_hosp)
    elif es_aisl:
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=9, color='#8E44AD', fill=True, fill_color='#8E44AD',
            fill_opacity=0.8,
            popup=folium.Popup(popup, max_width=250)
        ).add_to(fg_aislados)
    else:
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=5, color=color, fill=True, fill_color=color,
            fill_opacity=0.6,
            popup=folium.Popup(popup, max_width=250)
        ).add_to(fg_normal)

folium.LayerControl().add_to(m4)
m4.save('outputs/mapa_aislados_hospitales.html')
print('✓ Mapa guardado: outputs/mapa_aislados_hospitales.html')
m4

In [ ]:
# ============================================================
# SECCIÓN 10: PANEL RESUMEN — FIGURA PARA EL REPORTE
# ============================================================
# Una sola figura con los resultados clave (para el paper LaTeX)

fig = plt.figure(figsize=(16, 12))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# --- A: Mapa espacial con KDE ---
ax_a = fig.add_subplot(gs[0, 0])
iztapalapa.plot(ax=ax_a, facecolor='#EEF0F4', edgecolor='#2C3E50', linewidth=1.5)
for tipo in COLOR_TIPO:
    sub = df[df['tipo_establecimiento']==tipo]
    ax_a.scatter(sub['longitud'], sub['latitud'],
                 s=8, color=COLOR_TIPO[tipo], alpha=0.6, label=tipo)
ax_a.set_title('(A) Distribución espacial', fontsize=11, fontweight='bold')
ax_a.axis('off')

# --- B: Barcode H0 (top 30) ---
ax_b = fig.add_subplot(gs[0, 1])
h0_top = h0_f[np.argsort(h0_f[:,1]-h0_f[:,0])[::-1][:30]]
for y_i, (b, d) in enumerate(h0_top):
    ax_b.barh(y_i, d-b, left=b, height=0.7,
              color='#2980B9', alpha=0.7, edgecolor='white', lw=0.2)
ax_b.set_title('(B) Barcode H₀', fontsize=11, fontweight='bold')
ax_b.set_xlabel('Radio (m)', fontsize=9)
ax_b.set_ylabel('Componentes', fontsize=9)
ax_b.tick_params(labelsize=8)
ax_b.grid(axis='x', alpha=0.3)

# --- C: Barcode H1 ---
ax_c = fig.add_subplot(gs[0, 2])
if len(h1_f) > 0:
    h1_top = h1_f[np.argsort(h1_f[:,1]-h1_f[:,0])[::-1][:30]]
    pers_top = h1_top[:,1]-h1_top[:,0]
    for y_i, (b, d) in enumerate(h1_top):
        ax_c.barh(y_i, d-b, left=b, height=0.7,
                  color='#E67E22',
                  alpha=0.5 + 0.5*(d-b)/pers_top.max(),
                  edgecolor='white', lw=0.2)
ax_c.set_title('(C) Barcode H₁ (vacíos)', fontsize=11, fontweight='bold')
ax_c.set_xlabel('Radio (m)', fontsize=9)
ax_c.set_ylabel('Ciclos', fontsize=9)
ax_c.tick_params(labelsize=8)
ax_c.grid(axis='x', alpha=0.3)

# --- D: Diagrama de persistencia ---
ax_d = fig.add_subplot(gs[1, 0])
lim_d = max(h0_f[:,1].max(), h1_f[:,1].max() if len(h1_f) else 0) * 1.05
ax_d.plot([0,lim_d],[0,lim_d],'k--',lw=1,alpha=0.4)
ax_d.scatter(h0_f[:,0], h0_f[:,1], s=20, color='#2980B9', alpha=0.6, label='H₀')
ax_d.scatter(h1_f[:,0], h1_f[:,1] if len(h1_f) else [], s=40,
             color='#E67E22', alpha=0.8, marker='D', label='H₁')
ax_d.set_title('(D) Diagrama de persistencia', fontsize=11, fontweight='bold')
ax_d.set_xlabel('Nacimiento (m)', fontsize=9)
ax_d.set_ylabel('Muerte (m)', fontsize=9)
ax_d.legend(fontsize=8)
ax_d.tick_params(labelsize=8)
ax_d.grid(alpha=0.25)

# --- E: Componentes conectadas vs radio ---
ax_e = fig.add_subplot(gs[1, 1])
ax_e.plot(radios_eval, n_componentes, color='#2980B9', lw=2)
ax_e.fill_between(radios_eval, n_componentes, alpha=0.15, color='#2980B9')
ax_e.set_xlabel('Radio (m)', fontsize=9)
ax_e.set_ylabel('# Componentes (log)', fontsize=9)
ax_e.set_title('(E) Evolución de componentes H₀', fontsize=11, fontweight='bold')
ax_e.set_yscale('log')
ax_e.grid(alpha=0.3)
ax_e.tick_params(labelsize=8)

# --- F: Comparación público vs privado (H1 persistencias) ---
ax_f = fig.add_subplot(gs[1, 2])
h1_priv_f = dgm_priv[1][dgm_priv[1][:,1] < np.inf]
h1_pub_f  = dgm_pub[1][dgm_pub[1][:,1]  < np.inf]
data_box  = []
labels_box= []
if len(h1_priv_f): data_box.append(h1_priv_f[:,1]-h1_priv_f[:,0]); labels_box.append('Privado')
if len(h1_pub_f):  data_box.append(h1_pub_f[:,1]-h1_pub_f[:,0]);   labels_box.append('Público')
if data_box:
    bp = ax_f.boxplot(data_box, labels=labels_box, patch_artist=True,
                      medianprops=dict(color='black', lw=2))
    colors_box = ['#E74C3C', '#27AE60']
    for patch, c in zip(bp['boxes'], colors_box):
        patch.set_facecolor(c); patch.set_alpha(0.6)
ax_f.set_title('(F) Persistencia H₁ por sector', fontsize=11, fontweight='bold')
ax_f.set_ylabel('Persistencia (m)', fontsize=9)
ax_f.tick_params(labelsize=9)
ax_f.grid(axis='y', alpha=0.3)

fig.suptitle('Análisis de Persistencia Homológica — Servicios Médicos Iztapalapa, CDMX',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig(figures_dir / '19_panel_resumen_tda.png', dpi=200, bbox_inches='tight')
plt.show()
print('✓ Panel resumen guardado: 19_panel_resumen_tda.png')

---
## Conclusiones del Análisis Topológico

### H₀ — Conectividad del sistema de salud
- A un radio de ~161 m (mediana de distancia al vecino más cercano), la mitad de los establecimientos ya tiene un vecino conectado.
- La conectividad completa de Iztapalapa se alcanza alrededor de 750 m — distancia a pie razonable pero no caminable para todos los usuarios.
- Los **57 establecimientos aislados** (percentil 90) se concentran en la periferia sur/oriente de la alcaldía.

### H₁ — Vacíos de cobertura
- Los ciclos H₁ persistentes identifican **anillos de servicios con interior sin cobertura** — zonas donde el acceso es indirecto.
- Los ciclos más persistentes sugieren brechas territoriales reales, especialmente en zonas con menor densidad de consultorios.

### Sector Público vs Privado
- El **87.4% de los establecimientos son privados**, principalmente consultorios pequeños (0-5 personas).
- El sector público (71 establecimientos) muestra mayor dispersión en sus ciclos H₁, indicando cobertura más heterogénea.

### Recomendación de negocio / política pública
> Las zonas con ciclos H₁ de alta persistencia son candidatos prioritarios para la instalación de nuevos servicios públicos de salud general. Una intervención focalizada en estas áreas mejoraría la cobertura sin duplicar oferta existente.